In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Get feature name
featurenamefile = 'Feature_name.dat'
with open(featurenamefile) as file:
    feature_name = [line.rstrip() for line in file]
file.close()
ft_dict = {key: i for i, key in enumerate(feature_name)}
n_feature = len(feature_name)

In [38]:
# Dataframe
df_coapdos = pd.read_csv('./coap_dos/coap_dos_10x1MB_NAS_Traffic_21_Apr_2024_06_58.csv',header=None, names=feature_name)
df_mqttflood = pd.read_csv('./mqttPUBflood/mqttpub_flood_100x10MB_NAS_Traffic_21_Apr_2024_03_39.csv',header=None, names=feature_name)
df_pingflood = pd.read_csv('./pingflood/ping_flood_NAS_Traffic_17_Apr_2024_18_42.csv',header=None, names=feature_name)
df_portscan = pd.read_csv('./port_scan/portscan_NAS_Traffic_17_Apr_2024_16_50.csv',header=None, names=feature_name)
df_sqlmap = pd.read_csv('./sqlmap/sqlmap_NAS_Traffic_17_Apr_2024_16_07.csv',header=None, names=feature_name)
df_tcpsyn = pd.read_csv('./tcpflood/tcp_syn_NAS_Traffic_18_Apr_2024_14_47.csv',header=None, names=feature_name)
df_normal = pd.read_csv('./normal/normal_NAS_Traffic_20_Apr_2024_16_00.csv',header=None, names=feature_name)
df_normal['label'] = 0
df_coapdos['label'] = 1
df_mqttflood['label'] = 2
df_pingflood['label'] = 3
df_portscan['label'] = 4
df_sqlmap['label'] = 5
df_tcpsyn['label'] = 6

train_dfs = [df_normal, df_coapdos, df_mqttflood, df_pingflood, df_portscan, df_sqlmap, df_tcpsyn]

# evaluation dataframe
df_eval_coapdos = pd.read_csv('./coap_dos/coap_dos_eval_NAS_Traffic_21_Apr_2024_16_24.csv',header=None, names=feature_name)
df_eval_mqttflood = pd.read_csv('./mqttPUBflood/mqttpub_flood_100x100MB_NAS_Traffic_18_Apr_2024_15_42.csv',header=None, names=feature_name)
df_eval_pingflood = pd.read_csv('./pingflood/pingflood_spoofed_eval_NAS_Traffic_21_Apr_2024_02_50.csv',header=None, names=feature_name)
df_eval_portscan = pd.read_csv('./port_scan/portscan_10k_eval_NAS_Traffic_20_Apr_2024_21_57.csv',header=None, names=feature_name)
df_eval_sqlmap = pd.read_csv('./sqlmap/sqlmap_eval_NAS_Traffic_20_Apr_2024_21_13.csv',header=None, names=feature_name)
df_eval_tcpsyn = pd.read_csv('./tcpflood/tcpsyn_rand_src_eval_NAS_Traffic_20_Apr_2024_22_05.csv',header=None, names=feature_name)
df_eval_normal = pd.read_csv('./normal/normal_eval_NAS_Traffic_21_Apr_2024_05_01_99percent.csv',header=None, names=feature_name)
df_eval_normal['label'] = 0
df_eval_coapdos['label'] = 1
df_eval_mqttflood['label'] = 2
df_eval_pingflood['label'] = 3
df_eval_portscan['label'] = 4
df_eval_sqlmap['label'] = 5
df_eval_tcpsyn['label'] = 6

eval_dfs = [df_eval_normal, df_eval_coapdos, df_eval_mqttflood, df_eval_pingflood, df_eval_portscan, df_eval_sqlmap, df_eval_tcpsyn]


In [39]:
print(df_normal.shape)
print(df_coapdos.shape)
print(df_mqttflood.shape)
print(df_pingflood.shape)
print(df_portscan.shape)
print(df_sqlmap.shape)
print(df_tcpsyn.shape)

print(df_eval_normal.shape)
print(df_eval_coapdos.shape)
print(df_eval_mqttflood.shape)
print(df_eval_pingflood.shape)
print(df_eval_portscan.shape)
print(df_eval_sqlmap.shape)
print(df_eval_tcpsyn.shape)

(164683, 8)
(37410, 8)
(11107, 8)
(84694, 8)
(130005, 8)
(11268, 8)
(71613, 8)
(139371, 8)
(37541, 8)
(16208, 8)
(31708, 8)
(20007, 8)
(6273, 8)
(49998, 8)


In [40]:
# Drop columns
drop_columns = ['idx','level','session_id','ue_id','direction']

for df in train_dfs:
    df.drop(columns=drop_columns,inplace=True)
    df['packet_hex'] = df['packet_hex'].map(lambda x: x[2:])

for df in eval_dfs:
    df.drop(columns=drop_columns,inplace=True)
    df['packet_hex'] = df['packet_hex'].map(lambda x: x[2:])


# zero-ing time stamp
def zerotime(df):
    min_timestamp = df['timestamp'].min()
    df['timestamp']= df['timestamp'].apply(lambda x: x-min_timestamp)

for df in train_dfs:
    zerotime(df)

for df in eval_dfs:
    zerotime(df)

# scale only for portscan dataset 
def scale(df,ratio):
    df['timestamp']= df['timestamp'].apply(lambda x: x/ratio)

scale(df_portscan,10)

scale(df_eval_portscan,10)

In [41]:
import numpy as np
from collections import defaultdict

def dirichlet_skew_split(df, num_clients=7, alpha=0.5):
    from numpy.random import dirichlet

    proportions = dirichlet([alpha] * num_clients)
    proportions = (np.cumsum(proportions) * len(df)).astype(int)[:-1]
    x_splits = np.split(df, proportions)
    
    return x_splits




In [42]:
# example usage
# sample data size (1000 samples, 10 features)
num_samples = 1000
num_features = 10
# generate random data
X = np.random.rand(num_samples, num_features)
print(X.shape)

# split data into 3 clients with Dirichlet distribution
x_splits = dirichlet_skew_split(X, num_clients=7, alpha=0.5)

print("Data split into {} clients".format(len(x_splits)))
for i, x_split in enumerate(x_splits):
    print("Client {}: {} samples".format(i, len(x_split)))

(1000, 10)
Data split into 7 clients
Client 0: 447 samples
Client 1: 0 samples
Client 2: 185 samples
Client 3: 2 samples
Client 4: 259 samples
Client 5: 104 samples
Client 6: 3 samples


In [43]:
# random new time stamp for small dataset and merge them to the main dataset
def random_starttime(maindf:pd.DataFrame, smalldfs: (pd.DataFrame), min_starttime=None, max_starttime=None):
    '''
    merge small dataset in the big dataset intertwainly in random start time manner
    maindf : |--------------------------------------|
    smalldf:        |--------|     |----------|
    '''
    print()
    total_attack_time = 0
    for df in smalldfs:
        total_attack_time += df['timestamp'].max()
    print(f"total attack time: {total_attack_time/60000} minutes")

    total_time = maindf['timestamp'].iloc[-1]
    print(f"total time: {total_time/60000} minutes")

    n_split = len(smalldfs)
    chunk_time = total_time//n_split

    for i in range(0,len(smalldfs)):
        chunk_start_time = chunk_time*i
        attack_time = smalldfs[i]['timestamp'].iloc[-1]
        print(f"attack {i} duration: {attack_time/60000} minutes")
        spare_time = chunk_time - attack_time
        if spare_time > 0:
            attack_start_time = chunk_start_time + np.random.randint(spare_time)
        else:
            attack_start_time = chunk_start_time

        smalldfs[i]['timestamp'] = smalldfs[i]['timestamp'].apply(lambda x: x + attack_start_time)
        
    df_chunks = [maindf]+smalldfs
    merged_df = pd.concat(df_chunks,ignore_index=True)
    merged_df.sort_values('timestamp',inplace=True)
    merged_df.reset_index(drop=True,inplace=True)
    
    return merged_df

# Split the training dataset

# split normal dataset into 7 equal chunks
normal_dfs = np.array_split(df_normal, 7)
for df in normal_dfs:
    zerotime(df)

# split df_coapdos into 2 chunks with ratio 10%, 90%
split_index = int(len(df_coapdos) * 0.1)  # Calculate index for 10% mark
coapdos_dfs = np.split(df_coapdos, [split_index])  # Split at calculated index
for df in coapdos_dfs:
    zerotime(df)

# split df_mqttflood into 2 chunks with ratio 10%, 90%
split_index = int(len(df_mqttflood) * 0.1)
mqttflood_dfs = np.split(df_mqttflood, [split_index])
for df in mqttflood_dfs:
    zerotime(df)

# split df_pingflood into 2 chunks with ratio 10%, 90%
split_index = int(len(df_pingflood) * 0.1)
pingflood_dfs = np.split(df_pingflood, [split_index])
for df in pingflood_dfs:
    zerotime(df)

# split df_portscan into 2 chunks with ratio 10%, 90%
split_index = int(len(df_portscan) * 0.1)
portscan_dfs = np.split(df_portscan, [split_index])
for df in portscan_dfs:
    zerotime(df)

# split df_sqlmap into 2 chunks with ratio 10%, 90%
split_index = int(len(df_sqlmap) * 0.1)
sqlmap_dfs = np.split(df_sqlmap, [split_index])
for df in sqlmap_dfs:
    zerotime(df)

# split df_tcpsyn into 2 chunks with ratio 10%, 90%
split_index = int(len(df_tcpsyn) * 0.1)
tcpsyn_dfs = np.split(df_tcpsyn, [split_index])
for df in tcpsyn_dfs:
    zerotime(df)

small_dfs = [coapdos_dfs[0], mqttflood_dfs[0], pingflood_dfs[0], portscan_dfs[0], sqlmap_dfs[0], tcpsyn_dfs[0]]

# merge small_dfs into 1 chunk of normal dataset
# named as shared_df
shared_df = random_starttime(normal_dfs[0], small_dfs)

# Merge the remaining chunks of attack data with the remaining normal chunks
large_dfs = [coapdos_dfs[1], mqttflood_dfs[1], pingflood_dfs[1], portscan_dfs[1], sqlmap_dfs[1], tcpsyn_dfs[1]]
merged_dfs = []
for i in range(6):
    # Merge one large attack dataset with one normal chunk
    region_df = random_starttime(normal_dfs[i+1], [large_dfs[i]])
    merged_dfs.append(region_df)

# For evaluation dataset, merge all the attack dataset with the normal dataset
all_attacks_eval_dfs = [df_eval_coapdos, df_eval_mqttflood, df_eval_pingflood, df_eval_portscan, df_eval_sqlmap, df_eval_tcpsyn]
big_eval_df = random_starttime(df_eval_normal, all_attacks_eval_dfs)


/home/linhngn/.local/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)



total attack time: 1.9381666666666666 minutes
total time: 9.591416666666667 minutes
attack 0 duration: 0.3227833333333333 minutes
attack 1 duration: 0.17351666666666668 minutes
attack 2 duration: 0.15583333333333332 minutes
attack 3 duration: 1.0833666666666666 minutes
attack 4 duration: 0.08883333333333333 minutes
attack 5 duration: 0.11383333333333333 minutes

total attack time: 2.0971333333333333 minutes
total time: 8.832 minutes
attack 0 duration: 2.0971333333333333 minutes

total attack time: 1.8511666666666666 minutes
total time: 8.7541 minutes
attack 0 duration: 1.8511666666666666 minutes

total attack time: 1.51675 minutes
total time: 8.752083333333333 minutes
attack 0 duration: 1.51675 minutes

total attack time: 9.755468333333333 minutes
total time: 8.681133333333333 minutes
attack 0 duration: 9.755468333333333 minutes

total attack time: 1.1758333333333333 minutes
total time: 8.74155 minutes
attack 0 duration: 1.1758333333333333 minutes

total attack time: 1.9195 minutes
to

In [44]:
# Save shared_df
shared_df.to_csv('public_shared_dataset.csv', index=False)

# Save merged_dfs
for i, merged_df in enumerate(merged_dfs):
    merged_df.to_csv(f"private_region_dataset_{i+1}.csv", index=False)

# save evaluation dataset
big_eval_df.to_csv('Combined_evaluation_dataset.csv', index=False)